# Turkish LLM Model Merging - Full Pipeline

Bu notebook, Turkce acik kaynak LLM modellerini **SLERP**, **TIES** ve **DARE** merge stratejileriyle birlestirir, benchmark ile karsilastirir ve HuggingFace'e yukler.

**Modeller:**
- Model A: `ytu-ce-cosmos/Turkish-Llama-8b-Instruct-v0.1`
- Model B: `Trendyol/Trendyol-LLM-8b-chat-v2.0`
- Base: `meta-llama/Meta-Llama-3-8B`

**Gereksinimler:**
- Google Colab **T4 GPU** (ucretsiz tier yeterli)
- HuggingFace token (Colab Secrets: `HF_TOKEN`)
- GitHub Personal Access Token (Colab Secrets: `GITHUB_TOKEN`)

---

## [Section 0] Kurulum
Bu section ~5 dakika surer

In [ ]:
!pip install -q mergekit transformers accelerate bitsandbytes huggingface_hub datasets pyyaml sentencepiece protobuf tqdm tabulate
print('Kutuphaneler kuruldu.')

In [ ]:
from google.colab import drive

MOUNT_DRIVE = True

if MOUNT_DRIVE:
    drive.mount('/content/drive')
    import os
    os.makedirs('/content/drive/MyDrive/turkish_merges/slerp', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/turkish_merges/ties', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/turkish_merges/dare', exist_ok=True)
    print('Google Drive baglandi.')
else:
    print('Drive mount atlanildi.')

In [ ]:
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF_TOKEN okundu.')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    if not HF_TOKEN:
        print('HF_TOKEN bulunamadi! Colab Secrets a ekleyin.')

try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
    print('GITHUB_TOKEN okundu.')
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')
    if not GITHUB_TOKEN:
        print('GITHUB_TOKEN bulunamadi! Colab Secrets a ekleyin.')

In [ ]:
from huggingface_hub import login
login(token=HF_TOKEN)
print('HuggingFace giris yapildi.')

In [ ]:
import os
import subprocess

GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')
REPO_OWNER = 'CengizhanBayram'
REPO_NAME = 'experiment_of_merging'
CLONE_DIR = '/content/' + REPO_NAME
PROJECT_DIR = CLONE_DIR

if GITHUB_TOKEN:
    clone_url = 'https://' + GITHUB_TOKEN + '@github.com/' + REPO_OWNER + '/' + REPO_NAME + '.git'
    print('GitHub token ile authenticated clone yapiliyor...')
else:
    clone_url = 'https://github.com/' + REPO_OWNER + '/' + REPO_NAME + '.git'
    print('Token yok, public olarak klonlaniyor...')

if not os.path.exists(CLONE_DIR):
    result = subprocess.run(['git', 'clone', clone_url, CLONE_DIR], capture_output=True, text=True)
    if result.returncode != 0:
        print('Clone HATASI: ' + result.stderr)
        print('Repo mevcut mu kontrol edin: https://github.com/' + REPO_OWNER + '/' + REPO_NAME)
    else:
        print('Clone basarili.')
else:
    print('Dizin zaten mevcut, pull yapiliyor...')
    subprocess.run(['git', '-C', CLONE_DIR, 'pull'])

os.chdir(CLONE_DIR)
print('Calisma dizini: ' + os.getcwd())

if os.path.isdir('scripts') and os.path.isdir('configs'):
    print('Proje dogrulandi.')
else:
    print('HATA: scripts/ veya configs/ bulunamadi! CWD=' + os.getcwd())

---
## [Section 1] Tokenizer Kontrolu
Bu section ~3 dakika surer

In [ ]:
os.chdir(PROJECT_DIR)
!python scripts/check_tokenizers.py

In [ ]:
import pandas as pd
from transformers import AutoTokenizer

models = [
    'ytu-ce-cosmos/Turkish-Llama-8b-Instruct-v0.1',
    'Trendyol/Trendyol-LLM-8b-chat-v2.0',
]

info_list = []
for m in models:
    try:
        tok = AutoTokenizer.from_pretrained(m, trust_remote_code=True)
        info_list.append({
            'Model': m.split('/')[-1],
            'Vocab Size': tok.vocab_size,
            'Full Vocab': len(tok),
            'Tokenizer': tok.__class__.__name__,
            'BOS': tok.bos_token,
            'EOS': tok.eos_token,
        })
    except Exception as e:
        print('Hata: ' + str(e))

df = pd.DataFrame(info_list)
display(df)

vocab_sizes = set(r['Full Vocab'] for r in info_list)
if len(vocab_sizes) > 1:
    print('UYARI: Vocab boyutlari farkli!')
else:
    print('Tum tokenizer lar uyumlu.')

---
## [Section 2] SLERP Merge
Bu section ~45 dakika surer (T4 GPU)

SLERP ile iki model birlestirilir, t=0.5 (esit agirlik)

In [ ]:
os.chdir(PROJECT_DIR)
!python scripts/run_merge.py --strategy slerp

In [ ]:
if MOUNT_DRIVE:
    import shutil
    from tqdm import tqdm
    src = os.path.join(PROJECT_DIR, 'merged_models', 'slerp')
    dst = '/content/drive/MyDrive/turkish_merges/slerp'
    if os.path.exists(src):
        files = [f for f in os.listdir(src) if os.path.isfile(os.path.join(src, f))]
        for f in tqdm(files, desc='SLERP to Drive'):
            shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
        print('SLERP modeli Drive a kopyalandi.')
    else:
        print('Kaynak bulunamadi: ' + src)

---
## [Section 3] TIES Merge
Bu section ~60 dakika surer (T4 GPU)

TIES ile iki model birlestirilir: weights=[0.6, 0.4], density=0.7

In [ ]:
os.chdir(PROJECT_DIR)
!python scripts/run_merge.py --strategy ties

In [ ]:
if MOUNT_DRIVE:
    import shutil
    from tqdm import tqdm
    src = os.path.join(PROJECT_DIR, 'merged_models', 'ties')
    dst = '/content/drive/MyDrive/turkish_merges/ties'
    if os.path.exists(src):
        files = [f for f in os.listdir(src) if os.path.isfile(os.path.join(src, f))]
        for f in tqdm(files, desc='TIES to Drive'):
            shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
        print('TIES modeli Drive a kopyalandi.')
    else:
        print('Kaynak bulunamadi: ' + src)

---
## [Section 4] DARE Merge
Bu section ~60 dakika surer (T4 GPU)

DARE-TIES ile iki model birlestirilir: weights=[0.6, 0.4], density=0.5

In [ ]:
os.chdir(PROJECT_DIR)
!python scripts/run_merge.py --strategy dare

In [ ]:
if MOUNT_DRIVE:
    import shutil
    from tqdm import tqdm
    src = os.path.join(PROJECT_DIR, 'merged_models', 'dare')
    dst = '/content/drive/MyDrive/turkish_merges/dare'
    if os.path.exists(src):
        files = [f for f in os.listdir(src) if os.path.isfile(os.path.join(src, f))]
        for f in tqdm(files, desc='DARE to Drive'):
            shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
        print('DARE modeli Drive a kopyalandi.')
    else:
        print('Kaynak bulunamadi: ' + src)

---
## [Section 5] Benchmark
Bu section ~90 dakika surer (T4 GPU, tum modeller sirayla)

In [ ]:
os.chdir(PROJECT_DIR)
!python scripts/benchmark.py --model all

In [ ]:
import json
import pandas as pd

results_path = os.path.join(PROJECT_DIR, 'results', 'benchmark_results.json')

if not os.path.exists(results_path):
    print('benchmark_results.json bulunamadi. Once benchmark calistirin.')
else:
    with open(results_path, 'r', encoding='utf-8') as f:
        results = json.load(f)

    rows = []
    for model_key, data in results.get('models', {}).items():
        names = {'slerp': 'SLERP', 'ties': 'TIES', 'dare': 'DARE', 'baseline': 'Baseline'}
        rows.append({
            'Model': names.get(model_key, model_key),
            'Perplexity': data.get('perplexity', 'N/A'),
            'Manuel Skor /20': data.get('manual_score', 'N/A'),
        })

    df = pd.DataFrame(rows)

    def highlight_best(s):
        styles = pd.DataFrame('', index=s.index, columns=s.columns)
        ppl = pd.to_numeric(s['Perplexity'], errors='coerce')
        if ppl.notna().any():
            styles.loc[ppl.idxmin(), 'Perplexity'] = 'background-color: #2ecc71; color: white; font-weight: bold'
        return styles

    display(df.style.apply(highlight_best, axis=None).set_caption('Benchmark Sonuclari'))
    print('Timestamp: ' + results.get('timestamp', 'N/A'))

In [ ]:
import json

results_path = os.path.join(PROJECT_DIR, 'results', 'benchmark_results.json')

if not os.path.exists(results_path):
    print('Sonuc dosyasi bulunamadi.')
else:
    with open(results_path, 'r', encoding='utf-8') as f:
        results = json.load(f)

    for model_key, data in results.get('models', {}).items():
        print('\n' + '='*70)
        print(model_key.upper() + ' - Detayli Yanitlar')
        print('='*70)
        responses = data.get('responses', {})
        if isinstance(responses, dict) and 'error' not in responses:
            for qid, resp in responses.items():
                emoji = '[+]' if resp.get('score', 0) == 1 else '[-]'
                print('  ' + emoji + ' [' + qid + '] ' + resp.get('question', '?'))
                print('     > ' + resp.get('response', 'N/A')[:200])
        else:
            print('  Yanit verisi yok: ' + str(responses))

---
## [Section 6] HuggingFace Push
Bu section ~30 dakika surer (model basina ~10 dakika)

In [ ]:
os.chdir(PROJECT_DIR)

SOURCE_MODELS_2 = 'ytu-ce-cosmos/Turkish-Llama-8b-Instruct-v0.1,Trendyol/Trendyol-LLM-8b-chat-v2.0'

push_configs = [
    {
        'model_path': './merged_models/slerp',
        'repo_id': 'Cosmobillian/TR-Llama-8B-Cosmos-Trendyol_SLERP_v1',
        'strategy': 'SLERP',
        'source_models': SOURCE_MODELS_2,
    },
    {
        'model_path': './merged_models/ties',
        'repo_id': 'Cosmobillian/TR-Llama-8B-3way_TIES_v1',
        'strategy': 'TIES',
        'source_models': SOURCE_MODELS_2,
    },
    {
        'model_path': './merged_models/dare',
        'repo_id': 'Cosmobillian/TR-Llama-8B-3way_DARE_v1',
        'strategy': 'DARE',
        'source_models': SOURCE_MODELS_2,
    },
]

push_results = []

for cfg in push_configs:
    if not os.path.exists(cfg['model_path']):
        print('Model bulunamadi: ' + cfg['model_path'] + ', atlaniyor.')
        push_results.append((cfg['repo_id'], 'EKSIK'))
        continue

    print('\n' + '='*70)
    print('Push: ' + cfg['repo_id'])
    print('='*70)

    cmd = (
        'python scripts/push_to_hub.py '
        + '--model_path ' + cfg['model_path'] + ' '
        + '--repo_id ' + cfg['repo_id'] + ' '
        + '--strategy ' + cfg['strategy'] + ' '
        + '--source_models "' + cfg['source_models'] + '"'
    )

    ret = os.system(cmd)

    if ret == 0:
        url = 'https://huggingface.co/' + cfg['repo_id']
        push_results.append((cfg['repo_id'], 'OK: ' + url))
        print('Basarili: ' + url)
    else:
        push_results.append((cfg['repo_id'], 'HATA'))
        print('Push basarisiz: ' + cfg['repo_id'])

print('\n' + '='*70)
print('PUSH OZETI')
print('='*70)
for repo, status in push_results:
    print('  ' + status + ' - ' + repo)

---
## Tamamlandi!

Tum merge, benchmark ve push islemleri tamamlandi.

**GitHub:** [CengizhanBayram/experiment_of_merging](https://github.com/CengizhanBayram/experiment_of_merging)